In [3]:
from trino_config import *
import pandas as pd
import pytz

In [4]:
pd.read_sql("show tables", trino)

,Table
0,sola_circuits
1,sola_partitions_lookup
2,sola_sites
3,sola_ts4


In [5]:
pd.read_sql("SELECT * FROM sola_ts4 limit 1", engine)

,circuit_id,t_stamp,power,energy,energy_reactive,energy_import,energy_export,energy_reactive_import,energy_reactive_export,power_factor,voltage,current,year,month,is_pv
0,11191,2024-12-01,-1.2933,-0.1078,-0.0069,None,None,None,None,0.99592,243.75,0.014,2024,12,False


In [2]:
df = pd.read_sql("SELECT * FROM sola_ts4 LIMIT 10", engine)
df.head()

,circuit_id,t_stamp,power,energy,energy_reactive,energy_import,energy_export,energy_reactive_import,energy_reactive_export,power_factor,voltage,current,year,month,is_pv
0,9281,2025-01-01 00:00:00,-2310.3600,-192.5300,2.0603,NaN,NaN,NaN,NaN,0.999885,243.70,9.4980,2025,1,True
1,6274,2025-01-01 00:00:00,-3286.2433,-273.8536,-3.3294,NaN,NaN,NaN,NaN,0.999852,244.00,13.4300,2025,1,True
2,290830,2025-01-29 04:40:00,3379.1767,281.5981,0.4303,281.5981,0.0,0.4828,0.0525,0.999998,245.00,13.7685,2025,1,True
3,5439,2025-01-01 00:00:00,468.5400,39.0450,-2.7117,NaN,NaN,NaN,NaN,0.995200,250.30,1.8665,2025,1,True
4,386268,2025-01-01 08:35:00,7.2700,0.6058,1.0719,0.6058,0.0,1.0719,0.0000,0.242086,236.05,0.1970,2025,1,True


In [8]:
pd.read_sql(f"select * from sola_partitions_lookup", engine)

,year,month,is_pv
0,2024,1,False
1,2024,1,True
2,2024,10,False
3,2024,10,True
4,2024,11,False
5,2024,11,True
6,2024,12,False
7,2024,12,True
8,2024,2,False
9,2024,2,True


In [3]:
pd.read_sql("select * from sola_sites  where site_id=425704325", engine)

,site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,inverter_count,pv_install_date,manufacturer,model,ac_capacity_kw_exploaded,installed_after_18_dec_2021
0,425704325,VIC,3631.0,145.4,-36.36,Powercor,250.0,226.0,None,2015-11-06,1000.0,2015-10-19,Enphase,M 215-60-240-S22 / 240-S23,0.226,False


In [4]:
pd.read_sql("select * from sola_circuits as c left join sola_sites as s on c.site_id = s.site_id where circuit_id=547781", engine)

,site_id,device_id,circuit_id,device_type,circuit_polarity,circuit_type,is_pv,site_id,state,postcode,...,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,inverter_count,pv_install_date,manufacturer,model,ac_capacity_kw_exploaded,installed_after_18_dec_2021
0,425704325,89006,547781,Watt Watcher,1,pv_site_net,True,425704325,VIC,3631.0,...,250.0,226.0,None,2015-11-06,1000.0,2015-10-19,Enphase,M 215-60-240-S22 / 240-S23,0.226,False


In [13]:
t0 = '2025-01-01 00:00:00+00:00'
t1 = '2025-01-03 00:00:00+00:00'

In [17]:
pd.read_sql(f"""select circuit_id, t_stamp from SolA_ts4 where year =2024 and is_pv=True and month=1 limit 1""", engine)

,circuit_id,t_stamp
0,6066,2024-01-01


In [6]:
pd.read_sql(f"SELECT * \
                 from SolA_ts4 as ts left join SolA_circuits as c on ts.circuit_id = c.circuit_id left join SolA_sites as s on c.site_id = s.site_id \
      limit 1 ", engine)

,circuit_id,t_stamp,power,energy,energy_reactive,energy_import,energy_export,energy_reactive_import,energy_reactive_export,power_factor,...,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,inverter_count,pv_install_date,manufacturer,model,ac_capacity_kw_exploaded,installed_after_18_dec_2021
0,5441,2024-10-01,0.0,0.0,0.0,None,None,None,None,None,...,1.02,1.0,None,2015-07-20,1.0,2007-01-01,SMA,Sunny Boy SB 1100,1.0,False


In [ ]:
df = pd.read_sql(f"SELECT s.site_id, t_stamp,  avg(voltage) as avg_voltage, avg(power) as avg_power, avg(energy) as avg_energy \
                 from SolA_ts4 as ts left join SolA_circuits as c on ts.circuit_id = c.circuit_id left join SolA_sites as s on c.site_id = s.site_id \
    where ts.is_pv = True and month=1 and year=2025 and state='NSW'  \
       and t_stamp >= TIMESTAMP '{t0}' and t_stamp < TIMESTAMP '{t1}' \
       group by s.site_id, t_stamp ", engine)

In [6]:
df = pd.read_sql(f"SELECT s.site_id, t_stamp,  avg(voltage) as avg_voltage, avg(power) as avg_power, avg(energy) as avg_energy \
                 from SolA_ts5 as ts left join SolA_circuits as c on ts.circuit_id = c.circuit_id left join SolA_sites as s on c.site_id = s.site_id \
    where ts.is_pv = True and month=1 and year=2025 and state='NSW'  \
       and t_stamp >= TIMESTAMP '{t0}' and t_stamp < TIMESTAMP '{t1}' \
       group by s.site_id, t_stamp ", engine)

In [49]:
t0 = '2025-01-01 00:00:00+11:00'
t1 = '2025-02-01 00:00:00+11:00'
pd.read_sql((f"""
    select circuit_id, count( t_stamp) as t_stamp_count, min(t_stamp ) as min_t_stamp, max(t_stamp ) as max_t_stamp
    from SolA_ts4 
    where is_pv = True and month=1 and year=2025 and circuit_id = 547781 
 and t_stamp >= TIMESTAMP '{t0}'  and t_stamp  < TIMESTAMP '{t1}'
                           
    group by circuit_id 
    order by t_stamp_count desc, circuit_id desc
"""), engine)

,circuit_id,t_stamp_count,min_t_stamp,max_t_stamp
0,547781,8789,2025-01-01,2025-01-31 12:55:00


In [12]:

pd.read_sql((f"""
    select circuit_id, count(distinct t_stamp) as t_stamp_count
    from SolA_ts4 
    where is_pv = False and month=1 and year=2025 and circuit_id = 547781 
    group by circuit_id 
    order by t_stamp_count desc, circuit_id desc
"""), engine)


,circuit_id,t_stamp_count


In [14]:
pd.read_sql((f"""
    select *
    from SolA_ts5 
                            limit 1
"""), engine)


,circuit_id,t_stamp,power,energy,energy_reactive,energy_import,energy_export,energy_reactive_import,energy_reactive_export,power_factor,voltage,current,site_id,state,year,month,is_pv
0,6275,2025-01-01,2853.7433,237.8119,-15.3103,None,None,None,None,0.995872,244.6,11.658,1476431461,SA,2025,1,False


In [9]:
t0 = '2025-01-01 00:00:00+00:00'
t1 = '2025-02-01 00:00:00+00:00'
df = pd.read_sql((f"""
    select max(t_stamp) as max_t_stamp, min(t_stamp) as min_t_stamp from SolA_ts4
    where is_pv = False and t_stamp  >= TIMESTAMP '{t0}' and t_stamp  < TIMESTAMP '{t1}' 
"""), engine)
df

,max_t_stamp,min_t_stamp
0,2025-01-31 23:55:00,2025-01-01


In [3]:
t0 = '2025-01-01 00:00:00+00:00'
t1 = '2025-02-01 00:00:00+00:00'
df = pd.read_sql((f"""
    select max(t_stamp) as max_t_stamp, min(t_stamp) as min_t_stamp from SolA_ts5
    where is_pv = True
"""), engine)
df

,max_t_stamp,min_t_stamp
0,2025-01-31 23:55:00,2025-01-01


In [20]:
df['max_t_stamp']

0   2025-01-31 23:55:00
Name: max_t_stamp, dtype: datetime64[ns]

In [23]:
df = pd.read_sql((f"""
    select *
    from SolA_ts 
    where is_pv = True and month=1 and year=2024 and circuit_id = 547781 
                            and t_stamp >= TIMESTAMP '{t0}' and t_stamp < TIMESTAMP '{t1}'
    order by t_stamp desc
"""), engine)
df

,circuit_id,t_stamp,power,energy,energy_reactive,energy_import,energy_export,energy_reactive_import,energy_reactive_export,power_factor,voltage,current,year,month,is_pv


In [5]:
df = pd.read_sql("SELECT current_timestamp", engine)
print(df)

                             _col0
0 2025-08-08 03:47:59.478000+00:00


In [ ]:
df['t_stamp'] = pd.to_datetime(df['t_stamp'], utc=True)
df['t_stamp'] = df['t_stamp'].dt.tz_convert(pytz.FixedOffset(600))  # Convert to UTC+10

In [ ]:
df['t_stamp'].min(), df['t_stamp'].max()